In [ ]:
!pip install -qU \
    langchain \
    langchain-community \
    langchain-google-genai \
    langchain-chroma \
    langchain-text-splitters \
    chromadb \
    pypdf \
    docx2txt \
    pandas \
    openpyxl \
    requests

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.0/148.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.6/79.6 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 61.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 385.1/385.1 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 81.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 62.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 56.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.5/571

In [ ]:
from google.colab import userdata


In [ ]:
import os

In [ ]:
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

In [ ]:
from google.colab import files


In [ ]:
import zipfile

In [ ]:

uploaded = files.upload()

Saving documents_atlas_bank.zip to documents_atlas_bank.zip


In [ ]:
for filename in uploaded.keys():
    if filename.endswith(".zip"):
        with zipfile.ZipFile(filename, "r") as zip_ref:
            zip_ref.extractall(".")
        print(f"Décompressé : {filename}")

Décompressé : documents_atlas_bank.zip


In [ ]:
import os
for f in os.listdir("documents"):
    print(f)

faq_courtes.txt
reglement_credits.pdf
procedures_internes.docx
faq_bancaire.docx
produits_bancaires.csv
services_tarifs.xlsx


In [ ]:
import json
import pandas as pd


In [ ]:
from pathlib import Path


In [ ]:
from langchain_core.documents import Document


In [ ]:
from langchain_community.document_loaders import (
    PyPDFLoader,
    Docx2txtLoader,
    TextLoader,
    CSVLoader,
)

/tmp/ipykernel_5359/837437006.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import (


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool

In [ ]:
DOCUMENT_DIRECTORY = "./documents"
CHROMA_DIRECTORY = "./chroma_db"

In [ ]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    temperature=0
)

In [ ]:
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

In [ ]:
def load_pdf(file_path):
    loader = PyPDFLoader(str(file_path))
    return loader.load()



In [ ]:
def load_docx(file_path):
    loader = Docx2txtLoader(str(file_path))
    return loader.load()


In [ ]:
def load_txt(file_path):
    loader = TextLoader(str(file_path), encoding="utf-8")
    return loader.load()


In [ ]:
def load_csv(file_path):
    loader = CSVLoader(file_path=str(file_path), encoding="utf-8")
    return loader.load()


In [ ]:




def load_excel(file_path):
    """Charge un fichier Excel en transformant chaque ligne de chaque
    feuille en un Document distinct, avec le nom de la feuille en metadata."""
    excel_file = pd.ExcelFile(file_path)
    documents = []

    for sheet_name in excel_file.sheet_names:
        df = pd.read_excel(file_path, sheet_name=sheet_name)
        df = df.fillna("")

        for index, row in df.iterrows():
            content_lines = [f"{col}: {row[col]}" for col in df.columns]
            text = f"[Feuille: {sheet_name}]\n" + "\n".join(content_lines)

            documents.append(
                Document(
                    page_content=text,
                    metadata={
                        "source": str(file_path),
                        "sheet": sheet_name,
                        "row": index
                    }
                )
            )
    return documents

In [ ]:
def load_document(file_path):
    file_path = Path(file_path)
    extension = file_path.suffix.lower()

    if extension == ".pdf":
        return load_pdf(file_path)
    elif extension == ".docx":
        return load_docx(file_path)
    elif extension == ".txt":
        return load_txt(file_path)
    elif extension == ".csv":
        return load_csv(file_path)
    elif extension in [".xlsx", ".xls"]:
        return load_excel(file_path)
    else:
        print(f"Fichier non supporté : {file_path}")
        return []

In [ ]:
def load_all_documents(directory):
    documents = []
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)

    for file_path in directory.rglob("*"):
        if file_path.is_file():
            try:
                loaded = load_document(file_path)
                documents.extend(loaded)
                print(f"Chargé : {file_path} ({len(loaded)} doc(s))")
            except Exception as error:
                print(f"Erreur en chargeant {file_path} : {error}")

    return documents

In [ ]:
documents = load_all_documents(DOCUMENT_DIRECTORY)


Chargé : documents/faq_courtes.txt (1 doc(s))
Chargé : documents/reglement_credits.pdf (2 doc(s))
Chargé : documents/procedures_internes.docx (1 doc(s))
Chargé : documents/faq_bancaire.docx (1 doc(s))
Chargé : documents/produits_bancaires.csv (14 doc(s))
Chargé : documents/services_tarifs.xlsx (21 doc(s))


In [ ]:
print(f"\nTotal de documents chargés : {len(documents)}")
if not documents:
    raise ValueError("Aucun document trouvé dans le dossier documents/.")



Total de documents chargés : 40


In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)


In [ ]:
chunks = text_splitter.split_documents(documents)
print(f"Chunks créés : {len(chunks)}")

Chunks créés : 51


In [ ]:
vector_store = Chroma(
    collection_name="atlas_bank_documents",
    embedding_function=embeddings,
    persist_directory=CHROMA_DIRECTORY
)


In [ ]:
vector_store.add_documents(chunks)


['dd04cb91-4379-4b76-bf9d-2f7f566ea135',
 '57343d07-f7cc-4378-993b-ceeaf5686a06',
 'b5077dc0-2afe-4fae-ad9a-1ebb3240c740',
 '102b3adb-496c-4bce-8927-27c1cd6b85c0',
 '0864b343-677d-460d-a78b-288a3c9fdc60',
 '7ba9f1b8-e62b-41e4-9293-9d2f116869d6',
 '68e8da1c-3e67-4f46-83a0-5eec38b7445d',
 '904d08f3-d897-4a2c-be72-9df019e1e27b',
 '4add1540-1966-4e6d-bcc6-f7269642f78c',
 'b685b02e-2b1a-4625-b33f-bb4508f4a03e',
 '03b3ed64-c0f5-4ab1-9faf-b3c6b302d8e2',
 'ede4ae99-c2a3-437e-abb8-e09f6b2f0642',
 '73d4ac47-de77-4f68-923b-be58a7aacae1',
 'abca28f2-3922-49ff-a758-3937a630d0d1',
 'a2c70147-dce2-404b-ae2b-7020e740963b',
 'a2ae22de-4393-46cb-b90a-41620de52005',
 '8ebf692a-85c7-4dae-a930-62d4e48ef538',
 '46442fd4-1259-4277-85b4-79dae29705fc',
 '122b590f-a7fa-4096-999b-aec6e700b045',
 '5c2784d8-080e-47a9-840c-cda08ce86689',
 'ec29358a-1508-4255-9568-1556f5f87f8e',
 'd95483d9-ddc3-4600-a8c8-b6bae918dc00',
 'fb99e06f-10cb-4b80-8a33-29008eb34be3',
 '79f0d3a3-0bb6-4363-bdd1-f69b2df2b335',
 'c5a9a148-cfe8-

In [ ]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)


In [ ]:
test_results = retriever.invoke("Quel est le taux du crédit immobilier ?")


In [ ]:
for doc in test_results:
    print(doc.page_content[:200])

[Feuille: Taux de Crédits]
Type de crédit: Crédit Immobilier
Montant min (DT): 20000
Montant max (DT): 500000
Durée min (mois): 60
Durée max (mois): 300
Taux annuel min (%): 6.9
Taux annuel max (%): 8
[Feuille: Taux de Crédits]
Type de crédit: Crédit Immobilier
Montant min (DT): 20000
Montant max (DT): 500000
Durée min (mois): 60
Durée max (mois): 300
Taux annuel min (%): 6.9
Taux annuel max (%): 8
annuel fixe compris entre 8,5% et 11,9% selon le profil du client et la durée choisie.
2.2 Crédit immobilier
Destiné à l'achat, la construction ou la rénovation d'un logement principal ou secondaire. 
annuel fixe compris entre 8,5% et 11,9% selon le profil du client et la durée choisie.
2.2 Crédit immobilier
Destiné à l'achat, la construction ou la rénovation d'un logement principal ou secondaire. 


In [ ]:
@tool
def search_bank_documents(query: str) -> str:
    """
    Recherche dans les documents internes d'Atlas Bank
    (règlements de crédit, FAQ, procédures internes,
    catalogue de produits, tarifs et services).

    Utilise cet outil pour toute question sur les produits,
    conditions de crédit, frais, procédures ou FAQ de la banque.
    """
    retrieved_documents = retriever.invoke(query)

    if not retrieved_documents:
        return "Aucune information pertinente trouvée dans les documents."

    results = []
    for i, doc in enumerate(retrieved_documents, start=1):
        source = doc.metadata.get("source", "Source inconnue")
        results.append(f"DOCUMENT {i}\nSource : {source}\n\n{doc.page_content}")

    return "\n\n---\n\n".join(results)

In [ ]:
@tool
def simulate_loan_payment(amount_dt: float, annual_rate_percent: float, duration_months: int) -> str:
    """
    Simule la mensualité d'un crédit à taux fixe (amortissement constant).

    Utilise cet outil quand l'utilisateur demande de calculer ou simuler
    une mensualité de crédit, avec un montant, un taux et une durée donnés.
    Ne PAS utiliser cet outil pour connaître les taux officiels d'Atlas Bank :
    pour cela, utilise search_bank_documents.
    """
    try:
        monthly_rate = (annual_rate_percent / 100) / 12
        n = duration_months

        if monthly_rate == 0:
            monthly_payment = amount_dt / n
        else:
            monthly_payment = amount_dt * (monthly_rate * (1 + monthly_rate) ** n) / (
                (1 + monthly_rate) ** n - 1
            )

        total_paid = monthly_payment * n
        total_interest = total_paid - amount_dt

        return f"""Simulation de crédit :
Montant emprunté : {amount_dt:,.2f} DT
Taux annuel : {annual_rate_percent}%
Durée : {duration_months} mois

Mensualité estimée : {monthly_payment:,.2f} DT/mois
Coût total du crédit : {total_paid:,.2f} DT
Total des intérêts payés : {total_interest:,.2f} DT

(Simulation indicative — hors frais de dossier et assurance emprunteur.)"""

    except Exception as error:
        return f"Erreur de calcul : {error}"

In [ ]:
import requests

@tool
def search_wikipedia(topic: str) -> str:
    """
    Recherche une information générale sur Wikipédia.

    Utilise cet outil uniquement pour des questions générales
    qui ne concernent PAS les documents internes d'Atlas Bank
    (ex : définitions économiques générales, contexte macroéconomique).
    """
    try:
        search_response = requests.get(
            "https://fr.wikipedia.org/w/api.php",
            params={
                "action": "query",
                "list": "search",
                "srsearch": topic,
                "format": "json",
                "utf8": 1
            },
            timeout=10
        )
        search_response.raise_for_status()
        results = search_response.json().get("query", {}).get("search", [])

        if not results:
            return f"Aucun article Wikipédia trouvé pour '{topic}'."

        title = results[0]["title"]

        summary_response = requests.get(
            "https://fr.wikipedia.org/api/rest_v1/page/summary/" + title.replace(" ", "_"),
            timeout=10
        )
        summary_response.raise_for_status()
        summary = summary_response.json().get("extract", "Pas de résumé disponible.")

        return f"Article Wikipédia : {title}\n\n{summary}"

    except Exception as error:
        return f"Erreur API Wikipédia : {error}"

In [ ]:
tools = [
    search_bank_documents,
    simulate_loan_payment,
    search_wikipedia,
]

In [ ]:
SYSTEM_PROMPT = """
Tu es l'assistant IA officiel d'Atlas Bank.

Tu réponds aux questions des clients sur les produits bancaires,
les conditions de crédit, les frais, les procédures et la FAQ.

Règles importantes :
- Utilise TOUJOURS search_bank_documents pour toute question sur les produits,
  taux, conditions, frais ou procédures d'Atlas Bank.
- Utilise simulate_loan_payment uniquement si l'utilisateur demande un calcul
  de mensualité avec un montant, un taux et une durée précis.
- Utilise search_wikipedia uniquement pour des questions générales qui ne
  concernent pas les documents internes.
- Ne jamais inventer de taux, de montant ou de condition qui n'apparaît pas
  dans les documents récupérés.
- Si l'information n'est pas trouvée dans les documents, dis-le clairement
  et invite le client à contacter une agence ou le service client.
- Réponds toujours en français, de façon claire et professionnelle.
"""

In [ ]:
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=SYSTEM_PROMPT
)


In [ ]:
question = "Quels sont les documents nécessaires pour demander un crédit immobilier ?"


In [ ]:
response = agent.invoke({
    "messages": [{"role": "user", "content": question}]
})

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3719: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3719: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


In [ ]:
print("Question :", question)
print("\nRéponse :")
print(response["messages"][-1].content)

Question : Quels sont les documents nécessaires pour demander un crédit immobilier ?

Réponse :
[{'type': 'text', 'text': "Pour faire une demande de **crédit immobilier** auprès d'Atlas Bank, voici les documents requis :\n\n* **Pièce d'identité :** Carte d'identité nationale (CIN)\n* **Justificatifs de revenus :** 3 derniers bulletins de salaire\n* **Justificatif d'emploi :** Attestation de travail\n* **Justificatifs bancaires :** 3 derniers relevés bancaires\n* **Justificatifs liés au projet :** \n  * Compromis de vente ou devis\n  * Titre de propriété / plan\n\n---\n\n**Conditions d'éligibilité à rappeler :**\n* Être âgé entre 21 ans minimum et 65 ans maximum à l'échéance finale du crédit.\n* Justifier d'un revenu régulier et domicilié auprès d’Atlas Bank.\n* Avoir un taux d'endettement global ne dépassant pas 40 % du revenu net mensuel.\n* Ne pas figurer sur la liste des incidents de paiement de la Centrale des Risques.\n\nN'hésitez pas à vous rendre dans l'agence Atlas Bank la plus

In [ ]:
def print_answer(response):
    """Affiche proprement la réponse finale de l'agent, quel que soit son format."""
    content = response["messages"][-1].content
    if isinstance(content, str):
        print(content)
    else:
        for block in content:
            if isinstance(block, dict) and block.get("type") == "text":
                print(block["text"])


question2 = (
    "Je veux emprunter 30000 DT sur 48 mois pour un crédit consommation. "
    "Quel taux Atlas Bank applique-t-il, et peux-tu me calculer la mensualité "
    "avec le taux maximum ?"
)

response2 = agent.invoke({
    "messages": [{"role": "user", "content": question2}]
})

print("Question :", question2)
print("\nRéponse :")
print_answer(response2)

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3719: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3719: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3719: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Question : Je veux emprunter 30000 DT sur 48 mois pour un crédit consommation. Quel taux Atlas Bank applique-t-il, et peux-tu me calculer la mensualité avec le taux maximum ?

Réponse :
Pour un **Crédit Consommation** chez Atlas Bank, les conditions applicables sont les suivantes :

* **Taux d'intérêt annuel :** De **8,5 % à 11,9 % / an**
* **Montant possible :** De 3 000 DT à 40 000 DT
* **Durée possible :** De 6 à 60 mois

---

### Simulation avec le taux maximum (11,9 %)

Pour un emprunt de **30 000 DT** sur **48 mois** au taux maximum de **11,9 %** :

* **Mensualité estimée :** **788,54 DT / mois**
* **Total des intérêts :** 7 850,06 DT
* **Coût total du crédit :** 37 850,06 DT

*(Note : Cette simulation est fournie à titre indicatif et ne comprend pas les éventuels frais de dossier ni l'assurance emprunteur. N'hésitez pas à contacter une agence Atlas Bank pour obtenir une offre personnalisée.)*


In [ ]:
conversation = []


In [ ]:
while True:
    user_input = input("\nVous : ")

    if user_input.lower() in ["exit", "quit", "stop"]:
        print("Fin de la conversation.")
        break

    conversation.append({"role": "user", "content": user_input})

    response = agent.invoke({"messages": conversation})
    ai_message = response["messages"][-1]

    print("\nAssistant Atlas Bank :")
    print_answer(response)

    content = ai_message.content
    text_only = content if isinstance(content, str) else "".join(
        b["text"] for b in content if isinstance(b, dict) and b.get("type") == "text"
    )
    conversation.append({"role": "assistant", "content": text_only})


Vous : Quels sont les frais de tenue de compte ?


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3719: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3719: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



Assistant Atlas Bank :
Chez Atlas Bank, les frais de tenue de compte courant s'élèvent à **5 DT par mois**.

Vous : Et pour une carte Gold ?


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3719: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3719: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



Assistant Atlas Bank :
Pour la **Carte Gold**, les frais s'élèvent à **90 DT par an**. 

Il s'agit d'une carte premium renouvelable annuellement, qui inclut notamment des assurances voyage.

Vous : Explique-moi ce qu'est l'inflation


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3719: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3719: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



Assistant Atlas Bank :
L'**inflation** est une hausse générale et durable des prix des biens et des services dans une économie sur une période donnée. 

Voici ce qu'il faut retenir sur l'inflation :

1. **Perte de pouvoir d'achat** : Lorsque l'inflation augmente, une même somme d'argent permet d'acheter moins de choses qu'auparavant. On dit que la monnaie perd de sa valeur ou de son pouvoir d'achat.
2. **Principales causes** :
   * **Par la demande** : La demande de biens et services dépasse l'offre disponible.
   * **Par les coûts** : Les coûts de production des entreprises (matières premières, énergie, salaires) augmentent, ce qui les pousse à répercuter cette hausse sur leurs prix de vente.
   * **Monétaire** : Une création excessive de monnaie par rapport à la richesse produite dans le pays.
3. **Mesure** : L'inflation est généralement mesurée par l'**Indice des Prix à la Consommation (IPC)**, qui suit l'évolution du coût d'un "panier" représentatif de biens et services consommés 